# Pengambilan dan Pembersihan Data

Proyek: Analisis Sentimen, Clustering, dan Regresi Likes pada Postingan X tentang Timnas

Penanggung jawab: Muhammad Yafi Adyatma, 24051130006

Notebook ini menghasilkan `dataset_timnas.csv` dengan kolom `text`, `created_at`, dan `like_count`, ditambah berkas dokumentasi yang dibaca oleh aplikasi Streamlit. Jalankan notebook ini dari folder `notebooks/` karena semua path bersifat relatif.

## 1. Dokumentasi Sumber Data

| Item | Isi |
|---|---|
| Platform | X, sebelumnya Twitter |
| Topik | Sepak bola dan Tim Nasional Indonesia |
| Kata kunci | timnas indonesia, timnas, garuda, #TimnasDay, pssi |
| Bahasa | Indonesia, memakai filter `lang:id` |
| Metode | Crawling memakai tweet-harvest melalui notebook ini, atau dataset cadangan yang disetujui dosen |
| Keluaran | `data/processed/dataset_timnas.csv`, `data/processed/metadata_pengambilan.json`, `data/processed/cleaning_log.csv` |

Syarat menjalankan crawling:

1. Node.js versi 18 atau lebih baru sudah terpasang.
2. Memiliki auth token dari cookie akun X yang sedang login.
3. Token disimpan pada environment variable `X_AUTH_TOKEN` dan tidak boleh ditulis di notebook atau diunggah ke repositori.

Jika crawling tidak bisa dilakukan, letakkan dataset cadangan berformat CSV di `data/raw/dataset_timnas_mentah.csv`, lalu ubah `RUN_CRAWL` menjadi `False` pada sel pengambilan data.

## 2. Konfigurasi

In [2]:
import os
import re
import glob
import json
import shutil
import subprocess
from datetime import datetime

import pandas as pd
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

KEYWORDS = ["timnas indonesia", "timnas", "garuda", "#TimnasDay", "pssi"]
LIMIT_PER_KEYWORD = 150
MIN_ROWS = 150

# Pola kesesuaian topik. Kata "garuda" sangat ambigu (maskapai, Garuda Linux,
# Purana, karakter game), sehingga hanya diterima bila disertai konteks sepak bola.
POLA_INTI = r"timnas|tim nasional|pssi|sepak\s?bola|sepakbola"
POLA_KONTEKS_GARUDA = (r"timnas|pssi|indonesia|bola|football|fifa|aff|piala|liga|"
                       r"pemain|pelatih|gol|mendunia|sea games|kualifikasi|"
                       r"suporter|supporter|pertiwi")
POLA_KECUALI_GARUDA = (r"linux|puran|hindu|vishnu|maskapai|penerbangan|airline|"
                       r"flight|pesawat|terbang|bandara|tiket|warframe|distro|"
                       r"xfce|cachyos|wallpaper|garuda aunty|garuda os|"
                       r"motogp|pertaminaenduro|vr46|mandalika|"
                       r"boeing|airbus|flightradar|aviation|pk-g|\bga\d{3}\b|"
                       r"\bgia\d|\bcgk\b|\bhnd\b|\bnrt\b")
POLA_TOPIK = r"timnas|pssi|garuda|indonesia|timnasday|timnasjuara|sepakbola|sepak bola"

# Kata fungsi dan slang Indonesia untuk memvalidasi bahasa tweet.
KATA_FUNGSI_ID = set("""
yang dan di ke dari untuk pada dengan adalah ini itu akan atau juga karena
sebagai oleh dalam telah sudah saya kamu dia kami kita mereka anda nya ku mu aku
tidak bukan belum jangan ada tak bisa dapat harus saja lebih sangat agar
saat ketika jika maka bahwa para setelah sebelum antara terhadap bagi
seperti supaya pula masih hanya jadi siap lagi lain inget akhirnya gabung
gak nggak ngga ga gk udah udh banget bgt kalau kalo klo tapi tpi tp aja aje
dong kok lah pun sih iya nih deh yg dgn utk tdk jg dr gw gue emang cuma
biar buat sama kayak kaya gitu gini org waktu abis mulai sampai orang
diatas kecuali mayoritas usai antar
""".split())


def bahasa_indonesia(teks_clean):
    """True bila teks dianggap berbahasa Indonesia."""
    kata = re.findall(r"[a-z]+", str(teks_clean))
    if not kata:
        return False
    if any(w in KATA_FUNGSI_ID for w in kata):
        return True
    try:
        lang = detect(str(teks_clean))
    except Exception:
        lang = None
    if lang == "id":
        return True
    if len(kata) <= 6 and re.search(POLA_TOPIK, str(teks_clean)):
        return True
    return False

# Path relatif terhadap folder notebooks/
RAW_DIR = os.path.join("..", "data", "raw")
CRAWL_DIR = os.path.join(RAW_DIR, "crawl")
RAW_CSV = os.path.join(RAW_DIR, "dataset_timnas_mentah.csv")
PROCESSED_DIR = os.path.join("..", "data", "processed")
OUTPUT_CSV = os.path.join(PROCESSED_DIR, "dataset_timnas.csv")

os.makedirs(CRAWL_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [3]:
# cek package node dan npm, lalu install playwright
!node -v
!npm -v
!pip install -q playwright langdetect
!python -m playwright install --with-deps chromium

v24.14.0
11.12.0



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip
c:\Users\VivoBook\AppData\Local\Programs\Python\Python313\python.exe: No module named playwright


In [4]:
from getpass import getpass

twitter_auth_token = getpass("Twitter auth_token: ")
print("token terisi:", bool(twitter_auth_token))

token terisi: True


## 3. Pengambilan Data

Setiap kata kunci diambil dalam berkas CSV terpisah di folder `data/raw/crawl/`. Gabungan semuanya disimpan sebagai `data/raw/dataset_timnas_mentah.csv`. Kegagalan pada satu kata kunci dicatat sebagai kendala dan tidak menghentikan kata kunci lainnya.

In [5]:
RUN_CRAWL = True
METODE = "Crawling dengan tweet-harvest melalui notebook"

kendala = []
WAKTU_PENGAMBILAN = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def slug(teks):
    return re.sub(r"[^a-z0-9]+", "_", teks.lower()).strip("_")

if RUN_CRAWL:
    if not twitter_auth_token:
        raise RuntimeError("Auth token belum diisi, jalankan sel input token.")

    for kw in KEYWORDS:
        filename = f"{slug(kw)}.csv"
        keyword = f"{kw} lang:id"
        limit = LIMIT_PER_KEYWORD

        !npx -y tweet-harvest@latest -o "{filename}" -s "{keyword}" --tab "LATEST" -l {limit} --token {twitter_auth_token}

        sumber = os.path.join("tweets-data", filename)
        if os.path.exists(sumber):
            os.replace(sumber, os.path.join(CRAWL_DIR, filename))
            print(f"Berhasil: {kw}")
        else:
            kendala.append(f"Crawling gagal untuk kata kunci {kw}")
            print(f"Gagal: {kw}")
else:
    METODE = "Dataset cadangan yang disetujui dosen"

Tweet Harvest [v2.7.1]

Research by Helmi Satria
Use it for Educational Purposes only!

This script uses Chromium Browser to crawl data from Twitter with your Twitter auth token.
Please enter your Twitter auth token when prompted.

Note: Keep your access token secret! Don't share it with anyone else.
Note: This script only runs on your local device.


Opening twitter search page...


-- Scrolling... (1)
Filling in keywords: timnas indonesia

 (2)

Your tweets saved to: c:\Matkul Semester 5\Praktik Aplikasi Web\project\TimnasDataAnalysis\notebooks\tweets-data\timnas_indonesia.csv
Total tweets saved: 20

-- Scrolling... (1) (2) (3)

Your tweets saved to: c:\Matkul Semester 5\Praktik Aplikasi Web\project\TimnasDataAnalysis\notebooks\tweets-data\timnas_indonesia.csv
Total tweets saved: 40

-- Scrolling... (1)

Your tweets saved to: c:\Matkul Semester 5\Praktik Aplikasi Web\project\TimnasDataAnalysis\notebooks\tweets-data\timnas_indonesia.csv
Total tweets saved: 60

-- Scrolling... (1) (2) 

## 4. Memuat dan Menyeragamkan Kolom

Nama kolom hasil crawling berbeda antar alat, sehingga dipetakan ke nama kolom standar proyek.

In [4]:
frames = []
for path in sorted(glob.glob(os.path.join(CRAWL_DIR, "*.csv"))):
    bagian = pd.read_csv(path)
    bagian["kata_kunci"] = os.path.splitext(os.path.basename(path))[0]
    frames.append(bagian)

if frames:
    raw = pd.concat(frames, ignore_index=True)
    raw.to_csv(RAW_CSV, index=False, encoding="utf-8")
elif os.path.exists(RAW_CSV):
    # Dataset cadangan yang diletakkan langsung di data/raw
    raw = pd.read_csv(RAW_CSV)
else:
    raise FileNotFoundError("Tidak ada data mentah di data/raw.")

if "kata_kunci" not in raw.columns:
    raw["kata_kunci"] = "dataset_cadangan"

KANDIDAT = {
    "text": ["full_text", "text", "content"],
    "created_at": ["created_at", "date", "tweet_created_at"],
    "like_count": ["favorite_count", "like_count", "likes", "likeCount"],
    "repost_count": ["retweet_count", "repost_count", "retweets"],
    "reply_count": ["reply_count", "replies"],
    "url": ["tweet_url", "url"],
    "lang": ["lang", "language"],
}

def ambil_kolom(df, kandidat):
    for nama in kandidat:
        if nama in df.columns:
            return df[nama]
    return pd.Series([None] * len(df), index=df.index)

df = pd.DataFrame({k: ambil_kolom(raw, v) for k, v in KANDIDAT.items()})
df["kata_kunci"] = raw["kata_kunci"]

if df["text"].isna().all():
    raise ValueError("Kolom teks tidak ditemukan pada data mentah.")

print("Jumlah data mentah:", len(df))
df.head()

NameError: name 'glob' is not defined

## 5. Pembersihan Data

Urutan tahap: teks kosong, duplikat, pembuatan `text_clean`, kesesuaian topik, penyaringan bahasa asing, validasi `like_count`. Jumlah baris setiap tahap dicatat untuk laporan.

In [7]:
log = []

def catat(tahap, data):
    log.append({"tahap": tahap, "jumlah_baris": len(data)})

catat("Data mentah", df)

# Teks kosong dan nilai hilang
df = df.dropna(subset=["text"])
df = df[df["text"].astype(str).str.strip() != ""]
catat("Hapus teks kosong", df)

# Duplikat berdasarkan teks asli
df = df.drop_duplicates(subset="text")
catat("Hapus duplikat teks asli", df)

In [8]:
# Buat kolom teks bersih lebih dahulu agar dipakai untuk filter topik dan bahasa.
# Negasi seperti tidak dan bukan sengaja dipertahankan karena memengaruhi sentimen.
def bersihkan_teks(teks):
    teks = str(teks).lower()
    teks = re.sub(r"^rt\s+", "", teks)
    teks = re.sub(r"http\S+|www\.\S+", " ", teks)
    teks = re.sub(r"@\w+", " ", teks)
    teks = re.sub(r"[^a-z0-9\s]", " ", teks)
    return re.sub(r"\s+", " ", teks).strip()

df["text_clean"] = df["text"].apply(bersihkan_teks)

# Kesesuaian topik. Kata "garuda" sangat ambigu (maskapai, Garuda Linux,
# Purana, karakter game), sehingga hanya diterima bila disertai konteks sepak bola.
teks_kecil = df["text"].astype(str).str.lower()
inti = teks_kecil.str.contains(POLA_INTI, regex=True)
garuda = (teks_kecil.str.contains("garuda")
          & teks_kecil.str.contains(POLA_KONTEKS_GARUDA, regex=True)
          & ~teks_kecil.str.contains(POLA_KECUALI_GARUDA, regex=True))
df = df[inti | garuda].copy()
catat("Buang postingan tidak relevan", df)

# Penyaringan bahasa. Kolom "lang" dari X dipakai lebih dahulu (kode "in"/"id"
# berarti Indonesia), lalu dilengkapi deteksi bahasa untuk menangkap tweet
# Indonesia yang salah label atau ketika kolom lang tidak tersedia.
if "lang" in df.columns:
    bahasa_x = df["lang"].astype(str).str.lower().isin(["in", "id"])
else:
    bahasa_x = pd.Series(False, index=df.index)
df = df[bahasa_x | df["text_clean"].apply(bahasa_indonesia)].copy()
catat("Buang postingan bahasa asing", df)

In [9]:
df = df[df["text_clean"] != ""]
catat("Hapus teks bersih kosong", df)

df = df.drop_duplicates(subset="text_clean")
catat("Hapus duplikat teks bersih", df)

In [3]:
# Validasi like_count. Nilai tidak valid menjadi kosong dan tidak diganti dengan nol
like = pd.to_numeric(df["like_count"], errors="coerce")
like = like.where(like >= 0)
df["like_count"] = like.round().astype("Int64")

for kolom in ["repost_count", "reply_count"]:
    df[kolom] = pd.to_numeric(df[kolom], errors="coerce").round().astype("Int64")

# Waktu publikasi disimpan dalam WIB
waktu = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
df["created_at"] = waktu.dt.tz_convert("Asia/Jakarta").dt.strftime("%Y-%m-%d %H:%M:%S")

jumlah_like_kosong = int(df["like_count"].isna().sum())
jumlah_waktu_kosong = int(df["created_at"].isna().sum())
catat("Dataset akhir", df)

print("Like tidak valid:", jumlah_like_kosong)
print("Waktu tidak valid:", jumlah_waktu_kosong)
pd.DataFrame(log)

NameError: name 'pd' is not defined

## 6. Menyimpan Dataset dan Dokumentasi

In [2]:
kolom_akhir = ["text", "created_at", "like_count", "repost_count",
               "reply_count", "url", "kata_kunci", "lang", "text_clean"]
df = df[kolom_akhir].reset_index(drop=True)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

pd.DataFrame(log).to_csv(os.path.join(PROCESSED_DIR, "cleaning_log.csv"), index=False)

if len(df) < MIN_ROWS:
    kendala.append(
        f"Jumlah data akhir {len(df)} kurang dari target minimal {MIN_ROWS} postingan unik"
    )
if jumlah_like_kosong > 0:
    kendala.append(f"{jumlah_like_kosong} postingan memiliki like_count yang tidak valid")

# Isi catatan tambahan sendiri bila ada kendala lain selama pengambilan data
KENDALA_MANUAL = []

metadata = {
    "topik": "Sepak bola dan Tim Nasional Indonesia",
    "kata_kunci": KEYWORDS,
    "metode": METODE,
    "waktu_pengambilan": WAKTU_PENGAMBILAN,
    "jumlah_data_mentah": int(log[0]["jumlah_baris"]),
    "jumlah_data_akhir": int(len(df)),
    "kendala": kendala + KENDALA_MANUAL,
}
with open(os.path.join(PROCESSED_DIR, "metadata_pengambilan.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Dataset tersimpan: {OUTPUT_CSV}, {len(df)} baris")
df.head()

NameError: name 'df' is not defined

## 7. Ringkasan Hasil

In [13]:
print("Jumlah postingan unik:", len(df))
print("Target minimal tercapai:", len(df) >= MIN_ROWS)
print("Rentang waktu:", df["created_at"].min(), "sampai", df["created_at"].max())
print()
print(df["like_count"].describe())
print()
print(df["kata_kunci"].value_counts())

Jumlah postingan unik: 635
Target minimal tercapai: True
Rentang waktu: 2026-10-07 11:50:38 sampai 2026-10-11 15:57:12

count         635.0
mean      60.590551
std      578.999057
min             0.0
25%             0.0
50%             0.0
75%             1.0
max         11826.0
Name: like_count, dtype: Float64

kata_kunci
pssi                157
timnas              136
timnasday           136
garuda              107
timnas_indonesia     99
Name: count, dtype: int64


## 8. Dokumentasi Kendala

Kendala yang tercatat otomatis tersimpan pada `metadata_pengambilan.json` dan ditampilkan di tab Data. Kendala yang umum muncul pada tahap ini:

- Crawling membutuhkan auth token akun X yang bisa kedaluwarsa atau dibatasi saat dipakai berulang.
- Jumlah hasil per kata kunci dibatasi oleh `LIMIT_PER_KEYWORD` dan oleh batas pemuatan halaman pencarian X.
- Kata kunci yang luas seperti garuda memunculkan postingan tidak relevan (maskapai, Garuda Linux, Purana, karakter game), sehingga perlu filter konteks topik.
- Pencarian tanpa filter bahasa dapat menarik postingan berbahasa asing; crawl memakai operator `lang:id` dan cleaning menyaring ulang memakai kolom `lang` dari X.
- Banyak postingan berupa retweet atau salinan yang menyebabkan duplikat.
- Nilai `like_count` adalah jumlah saat data diambil, sehingga postingan baru cenderung memiliki likes lebih sedikit.

Tambahkan kendala lain yang benar-benar dialami pada daftar `KENDALA_MANUAL` di sel penyimpanan, lalu jalankan ulang sel tersebut.